# Week 50, Day 4: Connected components

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w050d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
adj = {1: [2], 2: [1, 3], 3: [2], 4: [5], 5: [4], 6: []}

label, count = {}, 0
for s in adj:
    if s in label:
        continue
    count += 1
    stack = [s]
    label[s] = count
    while stack:
        v = stack.pop()
        for w in adj[v]:
            if w not in label:
                label[w] = count
                stack.append(w)
print(count, label)

In [ ]:
from collections import deque, Counter

def components(adj):
    label = {}
    for s in adj:
        if s in label:
            continue
        label[s] = s                         # name each component after its first vertex
        queue = deque([s])
        while queue:
            v = queue.popleft()
            for w in adj[v]:
                if w not in label:
                    label[w] = s
                    queue.append(w)
    return label

friends = {"ana": ["ben"], "ben": ["ana", "cy"], "cy": ["ben"], "dee": ["eve"], "eve": ["dee"], "fay": []}
label = components(friends)
sizes = Counter(label.values())
print(label, dict(sizes), label["ana"] == label["cy"], label["ana"] == label["eve"])

In [ ]:
def islands(grid):
    rows, cols = len(grid), len(grid[0])
    seen, sizes = set(), []
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] != "#" or (r, c) in seen:
                continue
            seen.add((r, c))
            stack, size = [(r, c)], 0
            while stack:
                y, x = stack.pop()
                size += 1
                for ny, nx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
                    if 0 <= ny < rows and 0 <= nx < cols and grid[ny][nx] == "#" and (ny, nx) not in seen:
                        seen.add((ny, nx))
                        stack.append((ny, nx))
            sizes.append(size)
    return sizes

sea = ["##..#",
       "#...#",
       "..#..",
       "....#"]
print(islands(sea))

In [ ]:
class UnionFind:
    def __init__(self):
        self.parent, self.size = {}, {}

    def add(self, x):
        if x not in self.parent:
            self.parent[x], self.size[x] = x, 1

    def find(self, x):
        self.add(x)
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:                # path compression
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False                              # already connected
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra                          # smaller tree under the larger
        self.size[ra] += self.size[rb]
        return True

uf = UnionFind()
for a, b in [("ana", "ben"), ("cy", "dee"), ("ben", "cy")]:
    uf.union(a, b)
uf.add("eve")
print(uf.find("ana") == uf.find("dee"), uf.find("eve") == uf.find("ana"), uf.size[uf.find("ana")])

In [ ]:
class UnionFind:
    def __init__(self, items):
        self.parent = {x: x for x in items}
        self.size = {x: 1 for x in items}
        self.count = len(self.parent)

    def find(self, x):
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        self.count -= 1
        return True

cables = [(1, 2), (3, 4), (2, 3), (1, 4), (5, 6), (4, 5)]
uf = UnionFind(range(1, 7))
for a, b in cables:
    merged = uf.union(a, b)
    print(f"cable {a}-{b}: {'joins two networks' if merged else 'makes a loop'}, {uf.count} network(s)")

> **Key idea.** Connected components split an undirected graph's vertices into groups joined by paths. Label them by searching (BFS or DFS) from every unlabelled vertex: O(V + E) in total. Grid islands are components of an implicit graph. Union-find keeps components as trees of parent links: `find` follows parents to the root, `union` links two roots; with union by size and path compression each operation is effectively constant. It tracks components as edges arrive, counts them, and spots the first edge that closes a cycle. In directed graphs, weakly connected ignores directions; strongly connected components (everyone reaches everyone) squash into a DAG.

### Exercise: Labelling components

Write `components(adj)`: a list of components, each a sorted list of vertices, the list sorted by each component's smallest vertex. Use an explicit stack or a queue (the checks use a 20,000-vertex chain). Then `connected(adj, u, v)` and `isolated(adj)`: the sorted list of vertices with no edges.

In [ ]:
def components(adj):
    ___

def connected(adj, u, v):
    ___

def isolated(adj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_components():\n    \"components\"\n    adj = {1: [2], 2: [1, 3], 3: [2], 4: [5], 5: [4], 6: []}\n    assert components(adj) == [[1, 2, 3], [4, 5], [6]]\n    assert connected(adj, 1, 3) and not connected(adj, 3, 4) and isolated(adj) == [6]\n    assert components({}) == []\n    chain = {i: [j for j in (i - 1, i + 1) if 0 <= j < 20_000] for i in range(20_000)}\n    assert len(components(chain)) == 1\n    rng = random.Random(1)\n    for _ in range(100):\n        n = rng.randint(1, 10)\n        g = {v: [] for v in range(n)}\n        for _ in range(rng.randint(0, 8)):\n            if n > 1:\n                a, b = rng.sample(range(n), 2)\n                g[a].append(b)\n                g[b].append(a)\n        comps = components(g)\n        assert sorted(v for c in comps for v in c) == list(range(n))\n        for c in comps:\n            for v in c:\n                assert all(w in c for w in g[v])")

### Exercise: Islands

In a grid of `#` (land) and `.` (water), with land joined up, down, left and right only, write `island_sizes(grid)`: the sizes of all islands, largest first. Then `coastline(grid)`: for each island (in the same order), its perimeter, counting each side of a land cell that faces water or the edge of the grid. Then `lakes(grid)`: the number of water components that **don't** touch the edge of the grid.

In [ ]:
def island_sizes(grid):
    ___

def coastline(grid):
    ___

def lakes(grid):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_islands():\n    \"islands\"\n    sea = [\"##..#\",\n           \"#...#\",\n           \"..#..\",\n           \"....#\"]\n    assert island_sizes(sea) == [3, 2, 1, 1] and coastline(sea) == [8, 6, 4, 4]\n    assert island_sizes([\"....\"]) == [] and coastline([\"#\"]) == [4]\n    ring = [\"#####\",\n            \"#...#\",\n            \"#.#.#\",\n            \"#####\"]\n    assert lakes(ring) == 1 and island_sizes(ring) == [15] and lakes(sea) == 0")

### Exercise: Union-find

Write `UnionFind()` with `add(x)`, `find(x)` (adding x if new), `union(a, b)` (returning whether two components merged), `connected(a, b)`, `size_of(x)` (the size of x's component), `count` (an attribute: the number of components), and `groups()`: a list of sorted lists, sorted by smallest member. Use union by size and path compression, both **without recursion** (the checks build a chain of 50,000 unions).

In [ ]:
class UnionFind:
    def __init__(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef test_uf():\n    \"union-find\"\n    uf = UnionFind()\n    assert uf.union(\"ana\", \"ben\") and uf.union(\"cy\", \"dee\") and not uf.union(\"ben\", \"ana\")\n    uf.add(\"eve\")\n    assert uf.count == 3 and uf.connected(\"ana\", \"ben\") and not uf.connected(\"ana\", \"cy\")\n    uf.union(\"ben\", \"cy\")\n    assert uf.size_of(\"dee\") == 4 and uf.count == 2 and uf.groups() == [[\"ana\", \"ben\", \"cy\", \"dee\"], [\"eve\"]]\ndef test_random():\n    \"against labelling\"\n    rng = random.Random(2)\n    uf, edges = UnionFind(), []\n    for v in range(30):\n        uf.add(v)\n    for _ in range(40):\n        a, b = rng.sample(range(30), 2)\n        uf.union(a, b)\n        edges.append((a, b))\n        label = {v: v for v in range(30)}\n        changed = True\n        while changed:\n            changed = False\n            for x, y in edges:\n                m = min(label[x], label[y])\n                if label[x] != m or label[y] != m:\n                    label[x] = label[y] = m\n                    changed = True\n        for x in range(30):\n            for y in range(0, 30, 7):\n                assert uf.connected(x, y) == (label[x] == label[y])\n        assert uf.count == len(set(label.values()))\ndef test_big():\n    \"fast and flat\"\n    uf = UnionFind()\n    t0 = time.perf_counter()\n    for i in range(50_000):\n        uf.union(i, i + 1)\n    for i in range(50_000):\n        uf.find(i)\n    assert uf.count == 1 and uf.size_of(0) == 50_001 and time.perf_counter() - t0 < 3")

### Exercise: As the cables arrive

Cables between computers 1 to n arrive in order. Write `networks_after_each(n, cables)`: the number of separate networks after each cable; `all_connected_at(n, cables)`: the 1-based number of the cable after which every computer is connected (`None` if never; 0 if n ≤ 1); and `first_loop(cables)`: the first cable `(a, b)` that joins two computers already connected, or `None`. Use union-find (write a small one, or reuse yours).

In [ ]:
def networks_after_each(n, cables):
    ___

def all_connected_at(n, cables):
    ___

def first_loop(cables):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cables():\n    \"cables\"\n    cables = [(1, 2), (3, 4), (2, 3), (1, 4), (5, 6), (4, 5)]\n    assert networks_after_each(6, cables) == [5, 4, 3, 3, 2, 1]\n    assert all_connected_at(6, cables) == 6 and all_connected_at(7, cables) is None and all_connected_at(1, []) == 0\n    assert first_loop(cables) == (1, 4) and first_loop([(1, 2), (2, 3)]) is None")

### Exercise: Merging records

A sign-up list has records `(name, emails)`; two records belong to the same person when they share **any** email address (directly or through a chain of shared addresses). Write `merge_people(records)`: a list of `(name, sorted emails)`, one per person, using the name from the **first** record of each group, sorted by the first email. Join each record's emails with union-find over email addresses.

In [ ]:
def merge_people(records):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_merge():\n    \"merging\"\n    records = [(\"Ada\", [\"ada@x.org\", \"a.l@y.org\"]), (\"Ada L\", [\"a.l@y.org\", \"ada@z.net\"]),\n               (\"Bo\", [\"bo@x.org\"]), (\"Ada\", [\"ada@q.com\"]), (\"Bo Y\", [\"bo@x.org\", \"b@y.org\"])]\n    assert merge_people(records) == [\n        (\"Ada\", [\"a.l@y.org\", \"ada@x.org\", \"ada@z.net\"]),\n        (\"Ada\", [\"ada@q.com\"]),\n        (\"Bo\", [\"b@y.org\", \"bo@x.org\"]),\n    ]\n    assert merge_people([]) == [] and merge_people([(\"Cy\", [\"c@x\", \"c@x\"])]) == [(\"Cy\", [\"c@x\"])]\n    chain = [(f\"p{i}\", [f\"e{i}\", f\"e{i + 1}\"]) for i in range(200)]\n    assert merge_people(chain) == [(\"p0\", sorted(f\"e{i}\" for i in range(201)))]")

### Bug hunt: Linking the wrong things

This union links `a` directly under `b` instead of linking their **roots**, so when `a` already has a parent, that link is overwritten and the old group falls apart: this prints `False`. Link root to root so it prints `True`.

In [ ]:
parent = {x: x for x in "abcd"}

def find(x):
    while parent[x] != x:
        x = parent[x]
    return x

def union(a, b):
    parent[a] = b

union("a", "b")
union("a", "c")
print(find("b") == find("c"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"roots\"\n    expect_output(\"True\")")

### Bug hunt: Counting the same piece twice

This counts components but marks vertices as seen only when a search **starts** from them, so later vertices of a component already explored start searches of their own: this prints `5` for a graph with 3 components. Mark everything a search reaches, so it prints `3`.

In [ ]:
adj = {1: [2], 2: [1, 3], 3: [2], 4: [5], 5: [4], 6: []}
seen, count = set(), 0
for s in adj:
    if s in seen:
        continue
    seen.add(s)
    count += 1
    stack, local = [s], {s}
    while stack:
        v = stack.pop()
        for w in adj[v]:
            if w not in local:
                local.add(w)
                stack.append(w)
print(count)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"mark what's reached\"\n    expect_output(\"3\")")

### Bug hunt: Corners that touch

The problem says islands are joined up, down, left and right only, but this search also steps diagonally, so two separate islands that touch at a corner are counted as one: this prints `1`. Use the four straight directions so it prints `2`.

In [ ]:
grid = ["#.",
        ".#"]
seen, count = set(), 0
for r in range(2):
    for c in range(2):
        if grid[r][c] == "#" and (r, c) not in seen:
            count += 1
            seen.add((r, c))
            stack = [(r, c)]
            while stack:
                y, x = stack.pop()
                for dy in (-1, 0, 1):
                    for dx in (-1, 0, 1):
                        ny, nx = y + dy, x + dx
                        if 0 <= ny < 2 and 0 <= nx < 2 and grid[ny][nx] == "#" and (ny, nx) not in seen:
                            seen.add((ny, nx))
                            stack.append((ny, nx))
print(count)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"four directions\"\n    expect_output(\"2\")")

### Puzzle: Union by size

Arrange the lines into a union by size. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# Union-find on 1 to 6, with union by size (ties: the first argument's root stays the root),
# after union(1, 2), union(3, 4), union(2, 4), union(5, 6).

p = ___   # the number of components
q = ___   # the size of 3's component
r = ___   # True or False: 1 and 6 are connected
s = ___   # the root of 4's component
t = ___   # how many islands in ["#.#", "###", "..."], joining up, down, left, right

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 2 and q == 4 and r is False, \"p, q and r\"\n    assert s == 1 and t == 1, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: strongly connected components

Find the **strongly connected components** of a directed graph with **Kosaraju's algorithm** (S. Rao Kosaraju, 1978; published by Micha Sharir in 1981):

1. Run DFS over the whole graph and record the vertices in order of **finish** time (Wednesday's `finish_order`).
2. Reverse every edge.
3. Go through the vertices in **decreasing** finish time; from each one not yet assigned, run a DFS on the **reversed** graph: everything it reaches (and isn't assigned yet) is one SCC.

It works because the vertex finishing last lies in a component that nothing outside it can reach in the reversed graph, so the second search can't leak out of it. Write `scc(adj)` returning a list of components, each sorted, the list sorted by smallest vertex, using **explicit stacks** for both passes (the checks use a 20,000-vertex cycle). Then `condensation(adj)`: the DAG of components, as a dict from each component's index in `scc(adj)` to the sorted list of component indices it has edges to (no self-loops). Then `can_reach_everywhere(adj)`: whether the whole graph is one SCC.

In [ ]:
def scc(adj):
    ___

def condensation(adj):
    ___

def can_reach_everywhere(adj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef reach(adj, s):\n    seen, stack = {s}, [s]\n    while stack:\n        v = stack.pop()\n        for w in adj[v]:\n            if w not in seen:\n                seen.add(w)\n                stack.append(w)\n    return seen\ndef test_scc():\n    \"components\"\n    adj = {1: [2], 2: [3], 3: [1, 4], 4: [5], 5: [6], 6: [4], 7: [6, 8], 8: []}\n    assert scc(adj) == [[1, 2, 3], [4, 5, 6], [7], [8]]\n    assert condensation(adj) == {0: [1], 1: [], 2: [1, 3], 3: []}\n    assert not can_reach_everywhere(adj) and can_reach_everywhere({1: [2], 2: [1]}) and can_reach_everywhere({1: []})\ndef test_random():\n    \"against reachability\"\n    rng = random.Random(3)\n    for _ in range(100):\n        n = rng.randint(1, 9)\n        g = {v: [w for w in range(n) if w != v and rng.random() < 0.2] for v in range(n)}\n        r = {v: reach(g, v) for v in g}\n        want = sorted({tuple(sorted(w for w in g if w in r[v] and v in r[w])) for v in g})\n        assert scc(g) == [list(c) for c in want]\ndef test_big():\n    \"deep\"\n    n = 20_000\n    cycle = {i: [(i + 1) % n] for i in range(n)}\n    assert len(scc(cycle)) == 1 and can_reach_everywhere(cycle)")

## Warm-up

Three pieces: {1, 2, 3}, {4, 5} and the lonely 6. Each new search starts from a vertex no earlier search reached, and labels everything it finds.

## Labelling components

In an undirected graph, two vertices are in the same **connected component** when there's a path between them. Components split the vertices into separate groups: every vertex is in exactly one.

To find them all: go through the vertices; whenever one has no label yet, start a BFS or DFS from it and give everything it reaches a new label. Across all the searches, every vertex and edge is handled once, so the total is **O(V + E)**, however many components there are.

With labels, "are u and v connected?" is a dict lookup, and component sizes, the largest component, or the number of isolated vertices all follow at once.

## Islands

A grid of land and water is an implicit graph (Monday): land cells are vertices, joined to land cells up, down, left and right. Its components are **islands**. Counting them, finding the largest, or measuring each one's coastline are all component problems, solved by searching from each unlabelled land cell. (Whether diagonal neighbours count as connected is a choice the problem must state; here they don't.)

## Union-find

Labelling with a search works when the whole graph is known in advance. But often edges **arrive** one at a time (friendships made, cables laid, pixels joined) and after each you want to know: are these two connected now? How many pieces are there? Re-running a search after every edge costs O(V + E) each time.

A **union-find** structure (also called a **disjoint-set** structure) handles this. Each component is a small tree, stored as a dict from each vertex to its **parent**; the root of the tree names the component.

- `find(x)`: follow parents up to the root.
- `union(a, b)`: find both roots; if they differ, make one root the parent of the other. That's the whole merge.

Two improvements keep the trees very flat:

- **Union by size**: hang the smaller tree under the larger root, so trees stay shallow (height at most log₂ n).
- **Path compression**: after finding the root, point every vertex on the way directly at it, so the next find is immediate.

With both, a long run of operations costs so little per operation that it's effectively constant: the exact bound, proved by Robert Tarjan in 1975, involves the inverse Ackermann function, which is at most 4 for any input that could ever exist. The structure itself goes back to Bernard Galler and Michael Fischer in 1964.

## Edges that keep arriving

With union-find, a running count of components is easy: start with one per vertex, and subtract one each time `union` actually merges two. Two classic questions:

- **When does everything become connected?** Add edges in order until the count reaches 1.
- **Which edge first makes a cycle?** The first edge whose two ends already have the same root: it joins two vertices that were connected already. (That's also how next week's Kruskal's algorithm avoids cycles while building a minimum spanning tree.)

## Directed graphs

In a directed graph, "connected" has two meanings:

- **Weakly connected**: connected if you ignore the directions. Find these by treating every edge as two-way.
- **Strongly connected**: every vertex can reach every other **following** the directions. A one-way street system is usable only if it's strongly connected.

The **strongly connected components** (SCCs) of a directed graph are its maximal strongly connected groups. Squash each into a single vertex and what's left is always a DAG, which is why SCCs are a standard first step in analysing dependencies, web links and state machines. Today's challenge finds them with two depth-first searches.

## Practice

## Quiz

## Daily challenge

## Before you go

Look at a map of islands, or a photo of spots on a surface, and count the pieces by hand. Then think: how would you count them if the picture arrived one pixel at a time? Tomorrow: topological sort, putting dependent tasks in an order that respects every dependency.